---
title: Backpropagation
---

::: {.callout-tip}
## Slides
[Slides](https://docs.google.com/presentation/d/18J7G3N0W3XRP6AX0RHrYfOgiPV_2NHDUMjjXkqI_knc/edit?slide=id.g33a6271ded3_0_0#slide=id.g33a6271ded3_0_0)
::: 


When it comes to training models and finding the best parameters, there are <mark>two big challenges</mark> specific to neural networks: 

1) Since the largest models at the time of writing have $∼10^{12}$ parameters, we need a way to <mark>calculate the gradients $\frac{\partial L}{\partial \phi}$ eﬀiciently</mark>. 

    This is challenging not only because of the sheer number of parameters but also because the gradient needs to be computed for every parameter at every iteration of the training algorithm. 

2) <mark>Initializing the parameters</mark> for neural networks can be very tricky. If it is not done carefully, the <mark>initial losses and their gradients can be extremely large or small</small>. 

<!-- ## Problem definitions 

Consider a network $f[x,\phi]$ with multivariate input $\mathbf{x}$, parameters $\phi$, and three hidden
layers $\mathbf{h}_1,\mathbf{h}_2$, and $\mathbf{h}_3$:

\begin{eqnarray}
  \mathbf{h}_{1} &=& \mathbf{a}[\boldsymbol\beta_{0} +\boldsymbol\Omega_{0}\mathbf{x}]\nonumber \\
  \mathbf{h}_{2} &=& \mathbf{a}[\boldsymbol\beta_{1} +\boldsymbol\Omega_{1}\mathbf{h}_{1}] \nonumber\\
  \mathbf{h}_{3} &=& \mathbf{a}[\boldsymbol\beta_{2} +\boldsymbol\Omega_{2}\mathbf{h}_{2}] \nonumber\\
  \mbox{\bf f}[\mathbf{x},\boldsymbol\phi] &=& \boldsymbol\beta_{3} +\boldsymbol\Omega_{3}\mathbf{h}_{3},
 \end{eqnarray}

The model parameters $\phi= \{\boldsymbol\beta_0,\boldsymbol\Omega_0,\boldsymbol\beta_1,\boldsymbol\Omega_1,\boldsymbol\beta_2,\boldsymbol\Omega_2,\boldsymbol\beta_3,\boldsymbol\Omega_3\}$ consist of the bias
vectors $\boldsymbol\beta_k$ and weight matrices $\boldsymbol\Omega_k$ between every layer.

We also have individual loss terms $\ell_i$, which return the negative log-likelihood of the ground truth label $y_i$ given the model prediction $f[\mathbf{x}_i,\phi]$ for training input $\mathbf{x}_i$. 

For example, this might be the least squares loss $\ell_i = (f[\mathbf{x}_i,\phi]-y_i)^2$. The total loss is the
sum of these terms over the training data:

\begin{eqnarray}
  L[\boldsymbol\phi]= \sum_{i=1}^{I} \ell_{i}[\boldsymbol\phi].
 \end{eqnarray}

The most commonly used optimization algorithm for training neural networks is stochastic gradient descent (SGD), which updates the parameters as:


\begin{eqnarray}\label{eq:train2_sgd}
  \boldsymbol\phi_{t+1}\longleftarrow\boldsymbol\phi_{t} - \alpha \sum_{i\in\mathcal{B}_{t}}\frac{\partial \ell_{i}[\boldsymbol\phi_{t}]}{\partial \boldsymbol\phi},
 \end{eqnarray}

where $\alpha$ is the learning rate, and $\mathcal{B}_{t}$ contains the batch indices at iteration $t$. 

To compute
this update, we need to calculate the derivatives:

\begin{eqnarray}
  \frac{\partial \ell_{i}}{\partial\boldsymbol\beta_{k}} \quad\quad \mbox{and} \quad\quad \frac{\partial \ell_{i}}{\partial\boldsymbol\Omega_{k}},
 \end{eqnarray}

for the parameters {$\boldsymbol\beta_k$ ,$\boldsymbol\Omega_k$ } at every layer $k \in\{0,1,...,K\}$ and for each index $i$ in the batch. 

The first part of this chapter describes the backpropagation algorithm, which
computes these derivatives eﬀiciently. 

In the second part of the chapter, we consider how to initialize the network parameters
before we commence training. We describe methods to choose the initial weights $\boldsymbol\Omega_k$ and
biases $\boldsymbol\beta_k$ so that training is stable -->

<!-- ## Computing derivatives -->

As we saw previously, the derivatives of the loss  $\frac{\partial \ell_{i}}{\partial\boldsymbol\beta_{k}}$ and $\frac{\partial \ell_{i}}{\partial\boldsymbol\Omega_{k}}$ tell us how the loss changes when we make a small change to the parameters. Optimization algorithms exploit this information to manipulate the parameters so that the loss becomes smaller. 

<mark>The backpropagation algorithm computes these derivatives $\frac{\partial \ell_{i}}{\partial\boldsymbol\beta_{k}}$ and $\frac{\partial \ell_{i}}{\partial\boldsymbol\Omega_{k}}$</mark> in an efficient manner in two main phases: 1) Forward Pass and 2) Backward Pass.

<!-- The mathematical details are somewhat involved, so we first make two observations that provide some intuition. -->



## Forward Pass

During the forward pass, the network processes the input data and computes the activations of all hidden units layer by layer, from the input layer until the output is obtained. 

The activations at each layer are saved to be used during the second phase of backpropagation to compute the gradients efficiently.

<img src="assets/Chap07/Train2BPIntuitions.svg" style="filter: invert(1);" width="100%">

Note that the effects of any small change to the weight will be scaled by the activation of the source hidden unit. For example, the brown weight is applied to the second hidden unit at layer 1; if the activation of this unit doubles, then the effect of a small change to the brown weight will double too. Hence, to compute the derivatives of the weights, we need to calculate and store the activations at the hidden layers. This is known as the forward pass since it involves running the network equations sequentially.

## BackwardPass

During the backward pass, the network uses the stored activations from the forward pass to compute the gradients of the loss with respect to each parameter efficiently. This is done by propagating the error from the output layer back through the network to the input layer, hence the name "backpropagation."

<img src="assets/Chap07/Train2BPIntuitions2.svg" style="filter: invert(1);" width="100%">

Panel a (top): To compute how a change to a weight feeding into layer $h_3$ (brown arrow) changes the loss, we need to know how the hidden unit in $h_3$ changes the model output $f$ and how $f$ changes the loss (blue arrows). 

Panel b (center): To compute how a small change to a weight feeding into $h_2$ (brown arrow) changes the loss, we need to know (i) how the hidden unit in $h_2$ changes $h_3$, (ii) how $h_3$ changes $f$, and (iii) how $f$ changes the loss (blue arrows).

Panel c (bottom): Similarly, to compute how a small change to a weight feeding into $h_1$ (brown arrow) changes the loss, we need to know how $h_1$ changes $h_2$ and how these changes propagate through to the loss (blue arrows). The backward pass first computes derivatives at the end of the network and then works backward to exploit the inherent redundancy of these computations.

<!-- **Observation 1**: Each weight (element of $\boldsymbol\Omega_k$) multiplies the activation at a source hidden
unit and adds the result to a destination hidden unit in the next layer. 

It follows that the effect of any small change to the weight is amplified or attenuated by the activation at
the source hidden unit. 

Hence, we run the network for each data example in the batch and store the activations of all the hidden units. 

This is known as the forward pass. The stored activations will subsequently be used to compute the gradients.

**Observation 2**: A small change in a bias or weight causes a ripple effect of changes through the subsequent network. 

The change modifies the value of its destination hidden unit. This, in turn, changes the values of the hidden units in the subsequent layer, which will change the hidden units in the layer after that, and so on, until a change is made to
the model output and, finally, the loss.

Hence, to know how changing a parameter modifies the loss, we also need to know how changes to every subsequent hidden layer will, in turn, modify their successor. These same quantities are required when considering other parameters in the same or earlier layers. It follows that we can calculate them once and reuse them. For example, consider
computing the effect of a small change in weights that feed into hidden layers h3, h2,
and h1, respectively:

* To calculate how a small change in a weight or bias feeding into hidden layer h3
modifies the loss, we need to know (i) how a change in layer h3 changes the model
output f , and (ii) how a change in this output changes the loss ℓ (figure 7.2a).

• To calculate how a small change in a weight or bias feeding into hidden layer h2
modifies the loss, we need to know (i) how a change in layer h2 affects h3, (ii) how h3
changes the model output, and (iii) how this output changes the loss (figure 7.2b).

• To calculate how a small change in a weight or bias feeding into hidden layer h1
modifies the loss, we need to know (i) how a change in layer h1 affects layer h2,
(ii) how a change in layer h2 affects layer h3, (iii) how layer h3 changes the model
output, and (iv) how the model output changes the loss (figure 7.2c).

As we move backward through the network, we see that most of the terms we need
were already calculated in the previous step, so we do not need to re-compute them.
Proceeding backward through the network in this way to compute the derivatives is
known as the backward pass. -->


<!-- The ideas behind backpropagation are relatively easy to understand. However, the
derivation requires matrix calculus because the bias and weight terms are vectors and
matrices, respectively. To help grasp the underlying mechanics, the following section
derives backpropagation for a simpler toy model with scalar parameters. We then apply
the same approach to a deep neural network in section 7.4. -->





## Scalars example

Consider a model $f[x,\boldsymbol\phi]$ with eight scalar parameters $\boldsymbol\phi= \{\beta_0,\omega_0,\beta_1,\omega_1,\beta_2,\omega_2,\beta_3,\omega_3\}$ that consists of a composition of the functions sin[•],exp[•], and cos[•]:


\begin{eqnarray}
  \mbox{f}[x,\boldsymbol\phi] = \beta_3+\omega_3\cdot\cos\Bigl[\beta_2+\omega_2\cdot\exp\bigl[\beta_1+\omega_1\cdot\sin[\beta_0+\omega_0\cdot x]\bigr]\Bigr],
 \end{eqnarray}

and a least squares loss function $L[\boldsymbol\phi] = \sum_i \ell_i[\boldsymbol\phi]$ with individual terms:

\begin{eqnarray}
 \ell_i = (\mbox{f}[x_i,\boldsymbol\phi]-y_i)^2,
 \end{eqnarray}

where, as usual, $x_i$ is the $i^{th}$ training input, and $y_i$ is the $i^{th}$ training output. 

You can think of this as a simple neural network with one input, one output, one hidden unit at each layer, and different activation functions sin[•],exp[•], and cos[•] between each layer.

We aim to compute the derivatives:

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial \beta_{0}}, \quad \frac{\partial \ell_i}{\partial \omega_0}, \quad \frac{\partial \ell_i}{\partial \beta_{1}}, \quad \frac{\partial \ell_i}{\partial \omega_1}, \quad
 \frac{\partial \ell_i}{\partial \beta_{2}}, \quad \frac{\partial \ell_i}{\partial \omega_{2}}, \quad \frac{\partial \ell_i}{\partial \beta_{3}}, \quad\mbox{and} \quad \frac{\partial \ell_i}{\partial \omega_{3}}.
 \end{eqnarray}

Of course, we could find expressions for these derivatives by hand and compute them directly. 

However, some of these expressions are quite complex. For example:

\begin{eqnarray}\label{eq:train2_complicated_deriv}
 \frac{\partial \ell_i}{\partial \omega_{0}} &=& -2 \left( \beta_3+\omega_3\cdot\cos\Bigl[\beta_2+\omega_2\cdot\exp\bigl[\beta_1+\omega_1\cdot\sin[\beta_0+\omega_0\cdot x_i]\bigr]\Bigr]-y_i\right)\nonumber \\
 &&\hspace{0.5cm}\cdot \omega_1\omega_2\omega_3\cdot x_i\cdot\cos[\beta_0+\omega_0 \cdot x_i]\cdot\exp\Bigl[\beta_1 + \omega_1 \cdot \sin[\beta_0+\omega_0\cdot x_i]\Bigr]\nonumber\\
 && \hspace{1cm}\cdot \sin\biggl[\beta_2+\omega_2\cdot \exp\Bigl[\beta_1 + \omega_1 \cdot \sin[\beta_0+\omega_0\cdot x_i]\Bigr]\biggr].
 \end{eqnarray}

Such expressions are awkward to derive and code without mistakes and do not exploit the inherent redundancy; notice that the three exponential terms are the same.


The backpropagation algorithm is an eﬀicient method for computing all of these derivatives at once. 

It consists of (i) a forward pass, in which we compute and store a
series of intermediate values and the network output, and (ii) a backward pass, in which we calculate the derivatives of each parameter, starting at the end of the network, and
reusing previous calculations as we move toward the start.


### Forward pass

We treat the computation of the loss as a series of calculations:

\begin{eqnarray}
 f_{0} &=& \beta_{0} + \omega_{0}\cdot x_i\nonumber\\
 h_{1} &=& \sin[f_{0}]\nonumber\\
 f_{1} &=& \beta_{1} + \omega_{1}\cdot h_{1}\nonumber\\
 h_{2} &=& \exp[f_{1}]\nonumber\\
 f_{2} &=& \beta_{2} + \omega_{2} \cdot h_{2}\nonumber\\
 h_{3} &=& \cos[f_{2}]\nonumber\\
 f_{3} &=& \beta_{3} + \omega_{3}\cdot h_{3}\nonumber\\
 \ell_{i} &=& (f_{3}-y_{i})^2.
 \end{eqnarray}

<img src="assets/Chap07/Train2BP1.svg" style="filter: invert(1);" width="100%">

We compute and store the values of the intermediate variables $f_k$ and $h_k$ (figure above).

### Backward pass #1

In the first backward pass, we work backward from the end of the function computing the derivatives $\frac{\partial \ell_i}{\partial f_k}$ and $\frac{\partial \ell_i}{\partial h_k}$ of the loss with respect to the intermediate quantities. Each derivative is computed from the previous one by multiplying by terms of the form $\frac{\partial f_k}{\partial h_k}$ or $\frac{\partial h_k}{\partial f_{k-1}}$.

<br/>

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial f_{3}}, \quad \frac{\partial \ell_i}{\partial h_3}, \quad \frac{\partial \ell_i}{\partial f_2}, \quad
 \frac{\partial \ell_i}{\partial h_2}, \quad \frac{\partial \ell_i}{\partial f_1}, \quad \frac{\partial \ell_i}{\partial h_1}, \quad\mbox{and} \quad \frac{\partial \ell_i}{\partial f_0}.
 \end{eqnarray}

<br/>


 The first of these derivatives is straightforward:

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial f_{3}} = 2(f_3-y_i).
 \end{eqnarray}

The next derivative can be calculated using the chain rule:

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial h_{3}} =\frac{\partial f_{3}}{\partial h_{3}} \frac{\partial \ell_i}{\partial f_{3}} .
 \end{eqnarray}

The left-hand side asks how $\ell_i$ changes when $h_3$ changes. The right-hand side says we can
decompose this into (i) how $f_3$ changes when $h_3$ changes and (ii) how $\ell_i$ changes when $f_3$
changes. 

In the original equations, $h_3$ changes $f_3$, which changes $\ell_i$, and the derivatives
represent the effects of this chain. Notice that we already computed the second of these
derivatives, and the other is the derivative of $\beta_3 + \omega_3\cdot h_3$ with respect to $h_3$, which is $\omega_3$.
We continue in this way, computing the derivatives of the output with respect to
these intermediate quantities (figure below):

<img src="assets/Chap07/Train2BP2.svg" style="filter: invert(1);" width="100%">

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial f_{2}} &=& \frac{\partial h_{3}}{\partial f_{2}}\left(
 \frac{\partial f_{3}}{\partial h_{3}}\frac{\partial \ell_i}{\partial f_{3}} \right)
 \nonumber \\
 \frac{\partial \ell_i}{\partial h_{2}} &=& \frac{\partial f_{2}}{\partial h_{2}}\left(\frac{\partial h_{3}}{\partial f_{2}}\frac{\partial f_{3}}{\partial h_{3}}\frac{\partial \ell_i}{\partial f_{3}}\right)\nonumber \\
 \frac{\partial \ell_i}{\partial f_{1}} &=& \frac{\partial h_{2}}{\partial f_{1}}\left( \frac{\partial f_{2}}{\partial h_{2}}\frac{\partial h_{3}}{\partial f_{2}}\frac{\partial f_{3}}{\partial h_{3}}\frac{\partial \ell_i}{\partial f_{3}} \right)\nonumber \\
 \frac{\partial \ell_i}{\partial h_{1}} &=& \frac{\partial f_{1}}{\partial h_{1}}\left(\frac{\partial h_{2}}{\partial f_{1}} \frac{\partial f_{2}}{\partial h_{2}}\frac{\partial h_{3}}{\partial f_{2}}\frac{\partial f_{3}}{\partial h_{3}}\frac{\partial \ell_i}{\partial f_{3}} \right)\nonumber \\
 \frac{\partial \ell_i}{\partial f_{0}} &=& \frac{\partial h_{1}}{\partial f_{0}}\left(\frac{\partial f_{1}}{\partial h_{1}}\frac{\partial h_{2}}{\partial f_{1}} \frac{\partial f_{2}}{\partial h_{2}}\frac{\partial h_{3}}{\partial f_{2}}\frac{\partial f_{3}}{\partial h_{3}}\frac{\partial \ell_i}{\partial f_{3}} \right).\label{eq:train2_simple_chain}
 \end{eqnarray}

 In each case, we have already computed the quantities in the brackets in the previous
step, and the last term has a simple expression. In other words, we can reuse the previously computed derivatives
if we calculate them in reverse order.

### Backward pass #2

Finally, we consider how the loss $\ell_i$ changes when we change the
parameters $\{\beta_k\}$ and $\{\omega_k\}$. Once more, we apply the chain rule (figure 7.5):

\begin{eqnarray}
 \frac{\partial \ell_i}{\partial \beta_{k}} &=& \frac{\partial f_{k}}{\partial \beta_{k}}\frac{\partial \ell_i}{\partial f_{k}}\nonumber \\
 \frac{\partial \ell_i}{\partial \omega_{k}} &=& \frac{\partial f_{k}}{\partial \omega_{k}}\frac{\partial \ell_i}{\partial f_{k}}.
 \end{eqnarray}

\begin{eqnarray}
 \frac{\partial f_{k}}{\partial \beta_{k}} = 1 \quad\quad\mbox{and}\quad \quad \frac{\partial f_{k}}{\partial \omega_{k}} &=& h_{k}.
 \end{eqnarray}

<img src="assets/Chap07/Train2BP3.svg" style="filter: invert(1);" width="100%">

Backpropagation backward pass #2. Finally, we compute the derivatives
$\frac{\partial \ell_i}{\partial \beta_k}$ and $\frac{\partial \ell_i}{\partial \omega_k}$. Each derivative is computed by multiplying the
term $\frac{\partial \ell_i}{\partial f_k}$ by $\frac{\partial f_k}{\partial \beta_k}$ or $\frac{\partial f_k}{\partial \omega_k}$ as appropriate.

\begin{eqnarray}
 \frac{\partial f_{0}}{\partial \beta_{0}} = 1 \quad\quad\mbox{and}\quad \quad \frac{\partial f_{0}}{\partial \omega_{0}} &=& x_{i}.
 \end{eqnarray}

 Backpropagation is both simpler and more eﬀicient than computing the derivatives in-
dividually, as in equation below 



\begin{eqnarray}\label{eq:train2_complicated_deriv2}
 \frac{\partial \ell_i}{\partial \omega_{0}} &=& -2 \left( \beta_3+\omega_3\cdot\cos\Bigl[\beta_2+\omega_2\cdot\exp\bigl[\beta_1+\omega_1\cdot\sin[\beta_0+\omega_0\cdot x_i]\bigr]\Bigr]-y_i\right)\nonumber \\
 &&\hspace{0.5cm}\cdot \omega_1\omega_2\omega_3\cdot x_i\cdot\cos[\beta_0+\omega_0 \cdot x_i]\cdot\exp\Bigl[\beta_1 + \omega_1 \cdot \sin[\beta_0+\omega_0\cdot x_i]\Bigr]\nonumber\\
 && \hspace{1cm}\cdot \sin\biggl[\beta_2+\omega_2\cdot \exp\Bigl[\beta_1 + \omega_1 \cdot \sin[\beta_0+\omega_0\cdot x_i]\Bigr]\biggr].
 \end{eqnarray}



<!-- ```python

# import library
import numpy as np

def fn(x, beta0, beta1, beta2, beta3, omega0, omega1, omega2, omega3):
  return beta3+omega3 * np.cos(beta2 + omega2 * np.exp(beta1 + omega1 * np.sin(beta0 + omega0 * x)))

def loss(x, y, beta0, beta1, beta2, beta3, omega0, omega1, omega2, omega3):
  diff = fn(x, beta0, beta1, beta2, beta3, omega0, omega1, omega2, omega3) - y
  return diff * diff

beta0 = 1.0; beta1 = 2.0; beta2 = -3.0; beta3 = 0.4
omega0 = 0.1; omega1 = -0.4; omega2 = 2.0; omega3 = 3.0
x = 2.3; y = 2.0
l_i_func = loss(x,y,beta0,beta1,beta2,beta3,omega0,omega1,omega2,omega3)
print('l_i=%3.3f'%l_i_func)

dldbeta3_func = 2 * (beta3 +omega3 * np.cos(beta2 + omega2 * np.exp(beta1+omega1 * np.sin(beta0+omega0 * x)))-y)
dldomega0_func = -2 *(beta3 +omega3 * np.cos(beta2 + omega2 * np.exp(beta1+omega1 * np.sin(beta0+omega0 * x)))-y) * \
              omega1 * omega2 * omega3 * x * np.cos(beta0 + omega0 * x) * np.exp(beta1 +omega1 * np.sin(beta0 + omega0 * x)) *\
              np.sin(beta2 + omega2 * np.exp(beta1+ omega1* np.sin(beta0+omega0 * x)))
              
              dldomega0_fd = (loss(x,y,beta0,beta1,beta2,beta3,omega0+0.00001,omega1,omega2,omega3)-loss(x,y,beta0,beta1,beta2,beta3,omega0,omega1,omega2,omega3))/0.00001

print('dydomega0: Function value = %3.3f, Finite difference value = %3.3f'%(dldomega0_func,dldomega0_fd))

f1

# TODO compute all the f_k and h_k terms
# Replace the code below

f0 = beta0 + omega0 * x
h1 = np.sin(f0)
f1 = beta1 + omega1 * h1
h2 = np.exp(f1)
f2 = beta2 + omega2 * h2
h3 = np.cos(f2)
f3 = beta3 + omega3 * h3
l_i = loss(x, y, beta0, beta1, beta2, beta3, omega0, omega1, omega2, omega3)


# Let's check we got that right:
print("f0: true value = %3.3f, your value = %3.3f"%(1.230, f0))
print("h1: true value = %3.3f, your value = %3.3f"%(0.942, h1)) 
print("f1: true value = %3.3f, your value = %3.3f"%(1.623, f1))
print("h2: true value = %3.3f, your value = %3.3f"%(5.068, h2))
print("f2: true value = %3.3f, your value = %3.3f"%(7.137, f2))
print("h3: true value = %3.3f, your value = %3.3f"%(0.657, h3))
print("f3: true value = %3.3f, your value = %3.3f"%(2.372, f3))
print("l_i original = %3.3f, l_i from forward pass = %3.3f"%(l_i_func, l_i))

# TODO -- Compute the derivatives of the output with respect
# to the intermediate computations h_k and f_k (i.e, run the backward pass)
# I've done the first two for you.  You replace the code below:
dldf3 = 2* (f3 - y)
dldh3 = omega3 * dldf3
# Replace the code below
dldf2 = dldh3 * -np.sin(f2)
dldh2 = omega2 * dldf2
dldf1 = dldh2 * np.exp(f1)
dldh1 = omega1 * dldf1
dldf0 = dldh1 * np.cos(f0)

     

# Let's check we got that right
print("dldf3: true value = %3.3f, your value = %3.3f"%(0.745, dldf3))
print("dldh3: true value = %3.3f, your value = %3.3f"%(2.234, dldh3))
print("dldf2: true value = %3.3f, your value = %3.3f"%(-1.683, dldf2))
print("dldh2: true value = %3.3f, your value = %3.3f"%(-3.366, dldh2))
print("dldf1: true value = %3.3f, your value = %3.3f"%(-17.060, dldf1))
print("dldh1: true value = %3.3f, your value = %3.3f"%(6.824, dldh1))
print("dldf0: true value = %3.3f, your value = %3.3f"%(2.281, dldf0))

# TODO -- Calculate the final derivatives with respect to the beta and omega terms

dldbeta3 = dldf3
dldomega3 = dldf3 * h3
dldbeta2 = dldf2
dldomega2 = dldf2 * h2
dldbeta1 = dldf1
dldomega1 = dldf1 * h1
dldbeta0 = dldf0
dldomega0 = dldf0 * x

     

# Let's check we got them right
print('dldbeta3: Your value = %3.3f, True value = %3.3f'%(dldbeta3, 0.745))
print('dldomega3: Your value = %3.3f, True value = %3.3f'%(dldomega3, 0.489))
print('dldbeta2: Your value = %3.3f, True value = %3.3f'%(dldbeta2, -1.683))
print('dldomega2: Your value = %3.3f, True value = %3.3f'%(dldomega2, -8.530))
print('dldbeta1: Your value = %3.3f, True value = %3.3f'%(dldbeta1, -17.060))
print('dldomega1: Your value = %3.3f, True value = %3.3f'%(dldomega1, -16.079))
print('dldbeta0: Your value = %3.3f, True value = %3.3f'%(dldbeta0, 2.281))
print('dldomega0: Your value = %3.3f, Function value = %3.3f, Finite difference value = %3.3f'%(dldomega0, dldomega0_func, dldomega0_fd))
``` -->